### PART A

In [4]:
import os
import cv2
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def extract_image_features(image_path, target_size=(128, 128)):
    """
    Reads an image, preprocesses it, and extracts numerical features.
    """
    # 1. Read the image
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    # Inspect dimensions/channels (optional logging could go here)
    # original_shape = img.shape
    
    # 2. Resize to a common size and convert to grayscale
    img_resized = cv2.resize(img, target_size)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    total_pixels = target_size[0] * target_size[1]
    
    # 3. Extract Intensity Statistics (NumPy)
    mean_brightness = np.mean(img_gray)
    contrast = np.std(img_gray)  # Standard deviation as a measure of contrast
    
    # Define thresholds for dark and bright pixels (0-255 scale)
    dark_threshold = 50
    bright_threshold = 200
    
    dark_pixel_ratio = np.sum(img_gray < dark_threshold) / total_pixels
    bright_pixel_ratio = np.sum(img_gray > bright_threshold) / total_pixels
    
    # 4. Extract Edge-Based Features (OpenCV Canny)
    # Using Otsu's method to find dynamic thresholds could improve this, 
    # but fixed thresholds (e.g., 100, 200) work for a baseline.
    edges = cv2.Canny(img_gray, 100, 200)
    edge_density = np.sum(edges > 0) / total_pixels
    
    # Return as a feature vector (list)
    return [
        mean_brightness, 
        contrast, 
        dark_pixel_ratio, 
        bright_pixel_ratio, 
        edge_density
    ]

def build_dataset(dataset_dir):
    """
    Iterates through the dataset directory, extracts features, and assigns labels.
    """
    features = []
    labels = []
    
    # Updated to match the exact folder names from the image
    classes = {'NonCracks': 0, 'Cracks': 1}
    
    for class_name, label in classes.items():
        class_dir = os.path.join(dataset_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Warning: Directory not found - {class_dir}")
            continue
            
        for file_name in os.listdir(class_dir):
            file_path = os.path.join(class_dir, file_name)
            
            # Extract features for a single image
            img_features = extract_image_features(file_path)
            
            if img_features is not None:
                features.append(img_features)
                labels.append(label)
                
    # Create one feature row per image
    feature_names = ['Mean_Brightness', 'Contrast', 'Dark_Pixel_Ratio', 'Bright_Pixel_Ratio', 'Edge_Density']
    df = pd.DataFrame(features, columns=feature_names)
    df['Label'] = labels
    
    return df

def train_and_evaluate_model(X_train, X_test, y_train, y_test, model, model_name):
    """
    Trains a model, records execution times, and evaluates performance.
    """
    print(f"\n--- Evaluating {model_name} ---")
    
    # Training
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    # Prediction
    start_predict = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start_predict
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"Training Time: {train_time:.4f} seconds")
    print(f"Prediction Time: {predict_time:.4f} seconds")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)

# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    # 1. Path to your Mendeley Data extracted folders
    # Update this path to where your dataset is stored locally
    DATASET_DIRECTORY = "448" 
    
    # 2. Raw Image -> Preprocessing -> Feature Extraction
    print("Extracting features from raw images...")
    # Note: If running without the dataset present, this will return an empty DataFrame.
    df = build_dataset(DATASET_DIRECTORY)
    
    if not df.empty:
        # 3. Train-Test Split
        X = df.drop('Label', axis=1)
        y = df['Label']
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        # 4. Representation Improvement (Scaling)
        # Traditional models like SVMs rely heavily on distance calculations; scaling is critical.
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # 5. ML Model Training & Evaluation
        models = {
            "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
            "Support Vector Machine (RBF)": SVC(kernel='rbf', random_state=42)
        }
        
        for name, clf in models.items():
            train_and_evaluate_model(X_train_scaled, X_test_scaled, y_train, y_test, clf, name)
    else:
        print("Dataset not found or empty. Please ensure the path and folder structures are correct.")

Extracting features from raw images...

--- Evaluating Random Forest ---
Training Time: 0.2119 seconds
Prediction Time: 0.0194 seconds
Accuracy:  0.9500
Precision: 0.9500
Recall:    0.9500
F1-Score:  0.9500
Confusion Matrix:
 [[38  2]
 [ 2 38]]

--- Evaluating Support Vector Machine (RBF) ---
Training Time: 0.0000 seconds
Prediction Time: 0.0000 seconds
Accuracy:  0.9625
Precision: 0.9512
Recall:    0.9750
F1-Score:  0.9630
Confusion Matrix:
 [[38  2]
 [ 1 39]]


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Re-isolate features and labels directly from the dataframe
X = df.drop('Label', axis=1)
y = df['Label']

# 2. Re-split and scale the data locally in this cell
X_train_diag, X_test_diag, y_train_diag, y_test_diag = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled_diag = scaler.fit_transform(X_train_diag)

# 3. Train the model and extract importances
rf_diagnostic = RandomForestClassifier(n_estimators=100, random_state=42)
rf_diagnostic.fit(X_train_scaled_diag, y_train_diag)

print("Feature Importances:")
for name, importance in zip(X.columns, rf_diagnostic.feature_importances_):
    print(f"{name}: {importance:.4f}")

Feature Importances:
Mean_Brightness: 0.2507
Contrast: 0.0945
Dark_Pixel_Ratio: 0.2153
Bright_Pixel_Ratio: 0.2856
Edge_Density: 0.1539


### PART B

In [9]:
import pandas as pd
import numpy as np
import re
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def clean_text(text):
    """
    Performs basic text cleaning: lowercasing and removing non-alphabetic characters.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

def evaluate_text_representation(X_train_vec, X_test_vec, y_train, y_test, model, rep_name):
    """
    Trains a model on a specific text representation, evaluates it, and records execution times.
    """
    print(f"\n--- Results for {rep_name} ---")
    print(f"Number of generated features: {X_train_vec.shape[1]}")
    
    # Training
    start_train = time.time()
    model.fit(X_train_vec, y_train)
    train_time = time.time() - start_train
    
    # Prediction
    start_predict = time.time()
    y_pred = model.predict(X_test_vec)
    predict_time = time.time() - start_predict
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    print(f"Training Time:   {train_time:.4f} seconds")
    print(f"Prediction Time: {predict_time:.4f} seconds")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Precision:       {prec:.4f}")
    print(f"Recall:          {rec:.4f}")
    print(f"F1-Score:        {f1:.4f}")

# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    DATASET_PATH = "email_spam_classification/emails.csv" 
    
    try:
        df = pd.read_csv(DATASET_PATH)
        print("Dataset loaded successfully.")
        
        # 1. Isolate the target label
        # In this specific Kaggle dataset, the target is named 'Prediction'
        label_column = 'Prediction'
        y = df[label_column]
        
        print("\nClass Distribution (%):")
        print(y.value_counts(normalize=True) * 100)
        
        # 2. Reconstruct the raw text
        print("\nReconstructing raw text from pre-vectorized columns...")
        # Drop the ID column and the Target column to leave only the word columns
        word_matrix = df.drop(columns=['Email No.', 'Prediction'])
        word_columns = word_matrix.columns.values
        
        # Fast NumPy trick to repeat column names based on their cell counts
        reconstructed_texts = []
        for row_counts in word_matrix.values:
            # e.g., if 'the' has count 2, it adds ['the', 'the']
            words = np.repeat(word_columns, row_counts)
            reconstructed_texts.append(" ".join(words))
            
        df['cleaned_text'] = reconstructed_texts
        
        # 3. Train-Test Split
        X_train, X_test, y_train, y_test = train_test_split(
            df['cleaned_text'], y, test_size=0.2, random_state=42, stratify=y
        )
        
        # 4. Vectorization setup (Comparing two representations)
        # Representation A: Unigrams (single words)
        # Representation B: Unigrams + Bigrams (single words and two-word combinations)
        vectorizers = {
            "Representation A: Unigrams (CountVectorizer)": CountVectorizer(stop_words='english', ngram_range=(1, 1)),
            "Representation B: Unigrams + Bigrams (CountVectorizer)": CountVectorizer(stop_words='english', ngram_range=(1, 2))
        }
        
        # 5. Train and Compare
        model = MultinomialNB()
        
        for name, vec in vectorizers.items():
            print(f"\nVectorizing for {name}...")
            X_train_vec = vec.fit_transform(X_train)
            X_test_vec = vec.transform(X_test)
            
            evaluate_text_representation(X_train_vec, X_test_vec, y_train, y_test, model, name)
            
    except FileNotFoundError:
        print(f"Error: Could not find the dataset at '{DATASET_PATH}'.")

Dataset loaded successfully.

Class Distribution (%):
Prediction
0    70.99768
1    29.00232
Name: proportion, dtype: float64

Reconstructing raw text from pre-vectorized columns...

Vectorizing for Representation A: Unigrams (CountVectorizer)...

--- Results for Representation A: Unigrams (CountVectorizer) ---
Number of generated features: 2738
Training Time:   0.0218 seconds
Prediction Time: 0.0000 seconds
Accuracy:        0.9382
Precision:       0.8576
Recall:          0.9433
F1-Score:        0.8984

Vectorizing for Representation B: Unigrams + Bigrams (CountVectorizer)...

--- Results for Representation B: Unigrams + Bigrams (CountVectorizer) ---
Number of generated features: 54983
Training Time:   0.0000 seconds
Prediction Time: 0.0000 seconds
Accuracy:        0.9498
Precision:       0.8735
Recall:          0.9667
F1-Score:        0.9177


### PART C

In [10]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load and Reconstruct Data (matching your Part B)
df = pd.read_csv("email_spam_classification/emails.csv")
y = df['Prediction']
word_matrix = df.drop(columns=['Email No.', 'Prediction'])
word_columns = word_matrix.columns.values

reconstructed_texts = []
for row_counts in word_matrix.values:
    words = np.repeat(word_columns, row_counts)
    reconstructed_texts.append(" ".join(words))

X_train, X_test, y_train, y_test = train_test_split(
    reconstructed_texts, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Define Original vs. Improved Vectorizers
vectorizers = {
    "Original Representation B (Unrestrained Unigrams + Bigrams)": 
        CountVectorizer(stop_words='english', ngram_range=(1, 2)),
        
    "Improved Representation (Pruned Unigrams + Bigrams)": 
        CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=3, max_features=3000)
}

# 3. Train and Compare
model = MultinomialNB()

for name, vec in vectorizers.items():
    print(f"\n--- Results for {name} ---")
    
    # Vectorization
    X_train_vec = vec.fit_transform(X_train)
    X_test_vec = vec.transform(X_test)
    print(f"Number of generated features: {X_train_vec.shape[1]}")
    
    # Training
    start_train = time.time()
    model.fit(X_train_vec, y_train)
    train_time = time.time() - start_train
    
    # Metrics
    y_pred = model.predict(X_test_vec)
    print(f"Training Time:   {train_time:.4f} seconds")
    print(f"Accuracy:        {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1-Score:        {f1_score(y_test, y_pred):.4f}")


--- Results for Original Representation B (Unrestrained Unigrams + Bigrams) ---
Number of generated features: 54983
Training Time:   0.0035 seconds
Accuracy:        0.9498
F1-Score:        0.9177

--- Results for Improved Representation (Pruned Unigrams + Bigrams) ---
Number of generated features: 3000
Training Time:   0.0062 seconds
Accuracy:        0.9333
F1-Score:        0.8896
